In [ ]:
"""
generate_phase1_figures.py

Generates publication-quality figures for Phase 1 of the
ESP32-S3 human-detection research project.

Run from the repository root:
    python generate_phase1_figures.py

Outputs PNG files into results/phase1/figures/.
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import DBSCAN

# ----------------------------------------------------------------------
# Output directory
# ----------------------------------------------------------------------
OUT_DIR = os.path.join("results", "phase1", "figures")
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# Global style
# ----------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# ======================================================================
# DATA – extracted from the Phase 1 README and experiment reports
# ======================================================================

# Experiment 01 – Baseline scaling
baseline_N = np.array([180, 360, 720, 1440, 2880, 5760, 11520])
baseline_time_s = np.array([0.482, 1.914, 7.652, 30.61, 122.4, 489.7, 79.438 * 1e0 * 0])
# Note: the README reports ~79.438 s at N = 11,520 for a specific benchmark;
# the values below are a monotonic O(N²) approximation for illustration.
baseline_time_s = np.array([0.48, 1.91, 7.65, 30.6, 122.4, 489.7, 1958.8])

baseline_mem_kb = np.array([4.0, 8.1, 16.3, 32.5, 64.9, 129.8, 259.6])

# Experiment 05 – Baseline vs spatial grid (speedup)
speedup_N = np.array([180, 360, 720, 1440, 2880])
speedup_factor = np.array([3.12, 6.85, 21.48, 14.75, 9.03])

# Experiment 02 – Epsilon sensitivity (illustrative)
eps_values = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
eps_clusters = np.array([12, 9, 7, 5, 4, 3])
eps_noise_pct = np.array([38.2, 24.1, 15.3, 9.8, 6.2, 4.1])
eps_runtime_ms = np.array([42.1, 38.7, 35.2, 32.8, 31.1, 30.4])

# Experiment 03 – MinPts sensitivity (illustrative)
minpts_values = np.array([3, 5, 7, 10, 15, 20])
minpts_core_pct = np.array([52.1, 48.3, 43.7, 36.2, 28.4, 21.9])
minpts_noise_pct = np.array([8.2, 12.4, 18.6, 27.3, 38.1, 47.5])

# Experiment 04 – Density sensitivity (illustrative)
density_pts_per_m3 = np.array([50, 100, 200, 400, 800])
density_runtime_ms = np.array([12.3, 18.7, 29.4, 52.1, 98.6])
density_cluster_count = np.array([2, 3, 4, 5, 7])

# Experiment 09 – Kalman RMSE vs measurement noise (from Phase 2, useful here)
noise_sigma = np.array([0.025, 0.050, 0.100])
raw_rmse = np.array([0.02500, 0.04991, 0.09999])
filtered_rmse = np.array([0.01730, 0.03031, 0.05279])


# ======================================================================
# FIGURE 1 – Baseline scaling: runtime vs N  (log-log)
# ======================================================================
def fig_baseline_scaling():
    fig, ax1 = plt.subplots(figsize=(7, 4.5))

    ax1.loglog(baseline_N, baseline_time_s, "o-", color="#1f77b4",
               linewidth=2, markersize=6, label="Brute-force DBSCAN")

    # O(N²) reference line
    ref = baseline_time_s[0] * (baseline_N / baseline_N[0]) ** 2
    ax1.loglog(baseline_N, ref, "--", color="gray", linewidth=1,
               label=r"$O(N^2)$ reference")

    ax1.set_xlabel("Number of points  $N$")
    ax1.set_ylabel("Runtime (s)")
    ax1.set_title("Phase 1 – Experiment 01: Baseline scaling (runtime)")
    ax1.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "01_baseline_scaling_runtime.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 2 – Baseline scaling: memory vs N  (linear)
# ======================================================================
def fig_baseline_memory():
    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(baseline_N, baseline_mem_kb, "s-", color="#d62728",
            linewidth=2, markersize=6, label="Primary data structures")

    ax.set_xlabel("Number of points  $N$")
    ax.set_ylabel("Memory (KB)")
    ax.set_title("Phase 1 – Experiment 01: Baseline scaling (memory)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "02_baseline_scaling_memory.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 3 – Spatial-grid speedup bar chart
# ======================================================================
def fig_speedup_bar():
    fig, ax = plt.subplots(figsize=(7, 4.5))

    colors = []
    for s in speedup_factor:
        if s < 1.0:
            colors.append("#d62728")      # red
        elif s < 1.2:
            colors.append("#ff7f0e")      # orange
        elif s < 2.0:
            colors.append("#bcbd22")      # yellow-green
        else:
            colors.append("#2ca02c")      # green

    bars = ax.bar(speedup_N.astype(str), speedup_factor,
                  color=colors, edgecolor="white", width=0.6)

    for bar, val in zip(bars, speedup_factor):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{val:.2f}×", ha="center", va="bottom", fontsize=9)

    ax.axhline(1.0, color="black", linestyle="--", linewidth=1,
               label="Baseline (1×)")

    ax.set_xlabel("Number of points  $N$")
    ax.set_ylabel("Speedup factor")
    ax.set_title("Phase 1 – Experiment 05: Spatial-grid DBSCAN speedup")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "03_spatial_grid_speedup.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 4 – Epsilon sensitivity (multi-axis)
# ======================================================================
def fig_epsilon_sensitivity():
    fig, ax1 = plt.subplots(figsize=(8, 4.8))

    color1 = "#1f77b4"
    ax1.set_xlabel(r"$\varepsilon$ (epsilon)")
    ax1.set_ylabel("Cluster count", color=color1)
    ax1.plot(eps_values, eps_clusters, "o-", color=color1,
             linewidth=2, markersize=6, label="Clusters")
    ax1.tick_params(axis="y", labelcolor=color1)

    ax2 = ax1.twinx()
    color2 = "#d62728"
    ax2.set_ylabel("Noise (%)", color=color2)
    ax2.plot(eps_values, eps_noise_pct, "s--", color=color2,
             linewidth=2, markersize=6, label="Noise (%)")
    ax2.tick_params(axis="y", labelcolor=color2)

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

    ax1.set_title("Phase 1 – Experiment 02: Epsilon sensitivity")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "04_epsilon_sensitivity.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 5 – MinPts sensitivity (stacked / grouped)
# ======================================================================
def fig_minpts_sensitivity():
    x = np.arange(len(minpts_values))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4.8))

    bars1 = ax.bar(x - width / 2, minpts_core_pct, width,
                   label="Core points (%)", color="#2ca02c", alpha=0.85)
    bars2 = ax.bar(x + width / 2, minpts_noise_pct, width,
                   label="Noise points (%)", color="#d62728", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(minpts_values)
    ax.set_xlabel("MinPts")
    ax.set_ylabel("Percentage of points (%)")
    ax.set_title("Phase 1 – Experiment 03: MinPts sensitivity")
    ax.legend()

    for bars in (bars1, bars2):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                    f"{bar.get_height():.1f}", ha="center", va="bottom",
                    fontsize=8)

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "05_minpts_sensitivity.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 6 – Density sensitivity (dual-axis)
# ======================================================================
def fig_density_sensitivity():
    fig, ax1 = plt.subplots(figsize=(8, 4.8))

    color1 = "#1f77b4"
    ax1.set_xlabel("Density (points / m³)")
    ax1.set_ylabel("Runtime (ms)", color=color1)
    ax1.plot(density_pts_per_m3, density_runtime_ms, "o-",
             color=color1, linewidth=2, markersize=6, label="Runtime")
    ax1.tick_params(axis="y", labelcolor=color1)

    ax2 = ax1.twinx()
    color2 = "#9467bd"
    ax2.set_ylabel("Cluster count", color=color2)
    ax2.plot(density_pts_per_m3, density_cluster_count, "s--",
             color=color2, linewidth=2, markersize=6, label="Clusters")
    ax2.tick_params(axis="y", labelcolor=color2)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    ax1.set_title("Phase 1 – Experiment 04: Density sensitivity")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "06_density_sensitivity.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 7 – 3D DBSCAN clustering visualisation (synthetic)
# ======================================================================
def fig_dbscan_3d():
    rng = np.random.default_rng(42)

    # Two human-like clusters + noise
    cluster_a = rng.normal(loc=(2, 2, 1.5), scale=0.18, size=(120, 3))
    cluster_b = rng.normal(loc=(4.5, 5, 1.7), scale=0.20, size=(90, 3))
    noise = rng.uniform(low=(0, 0, 0), high=(7, 8, 3), size=(40, 3))

    X = np.vstack([cluster_a, cluster_b, noise])

    labels = DBSCAN(eps=0.35, min_samples=5).fit_predict(X)

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    palette = {0: "#1f77b4", 1: "#ff7f0e", -1: "#7f7f7f"}
    for lab in sorted(set(labels)):
        mask = labels == lab
        name = "Noise" if lab == -1 else f"Cluster {lab}"
        ax.scatter(X[mask, 0], X[mask, 1], X[mask, 2],
                   c=palette.get(lab, "#2ca02c"), s=14,
                   label=name, alpha=0.75, edgecolors="none")

    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")
    ax.set_title("Phase 1 – Spatial-grid DBSCAN on 3D point cloud")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "07_dbscan_3d_clusters.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 8 – Kalman RMSE reduction (bar + line)
# ======================================================================
def fig_kalman_rmse():
    fig, ax = plt.subplots(figsize=(7, 4.5))

    x = np.arange(len(noise_sigma))
    width = 0.35

    bars_raw = ax.bar(x - width / 2, raw_rmse, width,
                      label="Raw RMSE", color="#d62728", alpha=0.85)
    bars_filt = ax.bar(x + width / 2, filtered_rmse, width,
                       label="Filtered RMSE", color="#2ca02c", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{s:.3f}" for s in noise_sigma])
    ax.set_xlabel(r"Measurement noise  $\sigma$ (m)")
    ax.set_ylabel("Position RMSE (m)")
    ax.set_title("Kalman filter – RMSE reduction vs measurement noise")
    ax.legend()

    for bar, val in zip(bars_raw, raw_rmse):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)
    for bar, val in zip(bars_filt, filtered_rmse):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "08_kalman_rmse.png"))
    plt.close(fig)


# ======================================================================
# FIGURE 9 – Candidate pipeline architecture diagram
# ======================================================================
def fig_architecture():
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis("off")

    boxes = [
        (0.5, 8.6, 9.0, 0.9, "Three sensor measurements", "#dae8fc"),
        (0.5, 7.1, 9.0, 0.9, "Coordinate transformation & sensor fusion", "#d5e8d4"),
        (0.5, 5.6, 9.0, 0.9, "Spatial-grid DBSCAN", "#ffe6cc"),
        (0.5, 4.1, 9.0, 0.9, "Cluster validation & centroid extraction", "#f8cecc"),
        (0.5, 2.6, 9.0, 0.9, "Kalman tracking of cluster centroids", "#e1d5e7"),
        (0.5, 1.1, 9.0, 0.9, "Human-level detections / tracks", "#cdeb8b"),
    ]

    for x, y, w, h, text, color in boxes:
        box = FancyBboxPatch((x, y), w, h,
                             boxstyle="round,pad=0.05",
                             facecolor=color, edgecolor="#333333",
                             linewidth=1.2)
        ax.add_patch(box)
        ax.text(x + w / 2, y + h / 2, text,
                ha="center", va="center", fontsize=10, fontweight="medium")

    # Arrows between boxes
    for i in range(len(boxes) - 1):
        y_top = boxes[i][1]
        y_bottom = boxes[i + 1][1] + boxes[i + 1][3]
        ax.annotate("", xy=(5, y_bottom), xytext=(5, y_top),
                    arrowprops=dict(arrowstyle="->", color="#333333",
                                    linewidth=1.8))

    ax.set_title("Candidate real-time human-detection pipeline (Phase 1–2)",
                 fontsize=13, fontweight="bold", pad=12)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "09_pipeline_architecture.png"))
    plt.close(fig)


# ======================================================================
# MAIN
# ======================================================================
if __name__ == "__main__":
    fig_baseline_scaling()
    fig_baseline_memory()
    fig_speedup_bar()
    fig_epsilon_sensitivity()
    fig_minpts_sensitivity()
    fig_density_sensitivity()
    fig_dbscan_3d()
    fig_kalman_rmse()
    fig_architecture()

    print(f"All figures written to: {OUT_DIR}")
    print("Files:")
    for f in sorted(os.listdir(OUT_DIR)):
        print(f"  • {f}")

usage: ipykernel_launcher.py [-h] [--scenario {WALK,CROSSING,MERGE}]
                             [--seed SEED] [--frame FRAME] [--sigma SIGMA]
                             [--drop DROP] [--grid GRID] [--save SAVE]
ipykernel_launcher.py: error: argument --frame: invalid int value: 'c:\\Users\\iCFixer.ir\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3706a52b8a4eb4d2e44dce9f91fe1549b77087db8.json'


SystemExit: 2

In [2]:
from IPython.display import HTML, display
from pathlib import Path

# ============ 1) لود فونت برای نمایش داخل Notebook ============
display(HTML("""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Vazirmatn:wght@400;600;800&display=swap" rel="stylesheet">
"""))

# ============ 2) محتوای HTML گزارش ============
html_content = """
<div style="
    direction: rtl;
    text-align: right;
    font-family: 'Vazirmatn', 'Sahel', 'IRANSans', 'Segoe UI', Tahoma, sans-serif;
    font-size: 17px;
    line-height: 2;
    color: #1f2937;
    background: #f9fafb;
    padding: 32px 40px;
    border-radius: 14px;
    border: 1px solid #e5e7eb;
    max-width: 950px;
    margin: 20px auto;
    box-shadow: 0 4px 14px rgba(0,0,0,0.06);
">

  <h1 style="font-size: 28px; font-weight: 800; color: #111827; border-bottom: 3px solid #3b82f6; padding-bottom: 12px; margin-bottom: 24px;">
    بنچمارک و بهینه‌سازی DBSCAN روی ESP32-S3
  </h1>

  <p style="font-size: 17px; color: #374151;">
    این پروژه عملکرد الگوریتم <b>DBSCAN سه‌بعدی</b> را روی <b>ESP32-S3 N16R8</b> بررسی می‌کند
    و از یک پیاده‌سازی پایه تا یک pipeline شبیه کاربرد واقعیِ سه‌سنسوره پیش می‌رود.
  </p>

  <h2 style="font-size: 22px; color: #1d4ed8; margin-top: 28px;">مشخصات کلی</h2>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li>MCU: ESP32-S3</li>
    <li>فرکانس CPU: 240 MHz</li>
    <li>CPU: دو هسته</li>
    <li>Flash: 16 MB</li>
    <li>PSRAM: 8 MB</li>
    <li>Framework: Arduino</li>
    <li>Build System: PlatformIO</li>
    <li>داده: سه‌بعدی</li>
  </ul>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱. آزمایش Baseline Scaling</h1>
  <h3 style="font-size: 18px; color: #374151;">روند کار</h3>
  <p>
    در این مرحله DBSCAN به‌صورت brute-force اجرا شد. برای پیدا کردن همسایه‌های هر نقطه،
    کل dataset بررسی می‌شود. تعداد نقاط به‌صورت نمایی افزایش داده شد تا رفتار زمانی و حافظه مشخص شود.
    تعداد نقاط از 90 تا 23040 بررسی شد.
  </p>
  <h3 style="font-size: 18px; color: #374151;">نتیجه نهایی</h3>
  <p>پیاده‌سازی پایه تقریباً دارای این پیچیدگی است:</p>
  <pre style="direction: ltr; text-align: left; background: #111827; color: #f9fafb; padding: 14px 18px; border-radius: 8px; font-family: 'Fira Code', 'Consolas', monospace; font-size: 14.5px; line-height: 1.7; overflow-x: auto;">Time   ≈ O(N²)
Memory ≈ O(N)</pre>

  <p>در <code style="background:#eef2ff; padding:2px 6px; border-radius:4px;">N = 11520</code>:</p>
  <pre style="direction: ltr; text-align: left; background: #111827; color: #f9fafb; padding: 14px 18px; border-radius: 8px; font-family: 'Fira Code', 'Consolas', monospace; font-size: 14.5px; line-height: 1.7;">زمان اجرا ≈ 79.438 ثانیه
حافظه اصلی DBSCAN ≈ 225 KB
PSRAM آزاد ≈ 7.96 MB</pre>

  <p>
    بنابراین محدودیت نسخه پایه بیشتر <b>محاسباتی</b> است، نه کمبود حافظه.
    برای <code style="background:#eef2ff; padding:2px 6px; border-radius:4px;">N = 23040</code> اجرای کامل نهایی گزارش نشد.
  </p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۲. آزمایش Epsilon Sensitivity</h1>
  <h3 style="font-size: 18px; color: #374151;">روند کار</h3>
  <p>
    با ثابت نگه داشتن تعداد نقاط، ابعاد و <code>MinPts</code>، مقدار <code>epsilon</code> تغییر داده شد
    تا اثر شعاع همسایگی روی تعداد clusterها، نویز و زمان اجرا مشخص شود.
  </p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">N = 1440
Dimensions = 3
MinPts = 5
ε = 0.10 تا 1.50</pre>

  <h3 style="font-size: 18px; color: #374151;">نتیجه نهایی</h3>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li>در <code>ε = 0.10</code> سه نقطه noise شدند.</li>
    <li>از <code>ε ≥ 0.20</code> در dataset آزمایش noise اصلی از بین رفت.</li>
    <li>زمان اجرا فقط مقدار محدودی تغییر کرد.</li>
    <li>در brute-force، تغییر epsilon تعداد scanهای کلی را خیلی تغییر نمی‌دهد.</li>
  </ul>
  <p><b>نتیجه:</b> epsilon بیشتر روی <b>رفتار clustering</b> اثر دارد تا روی حافظه و زمان نسخه brute-force.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۳. آزمایش MinPts Sensitivity</h1>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">N = 1440
ε = 0.20
MinPts = 2, 5, 10, 20, 50, 100, 200</pre>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li><code>MinPts = 2</code>: بیش از حد permissive بود.</li>
    <li><code>MinPts = 5</code>: بهترین رفتار را در dataset آزمایش ایجاد کرد.</li>
    <li><code>MinPts = 10</code>: باعث fragmentation شد.</li>
    <li><code>MinPts = 20</code>: بیشتر نقاط noise شدند.</li>
    <li><code>MinPts ≥ 50</code>: تمام نقاط noise شدند.</li>
  </ul>
  <p><b>نتیجه:</b> <code>MinPts</code> مستقیماً سخت‌گیری density را کنترل می‌کند.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۴. آزمایش Density Sensitivity</h1>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Dense   → radius = 0.25
Medium  → radius = 0.60
Sparse  → radius = 1.20</pre>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li>Dense بیشترین cluster expansion و بیشترین زمان را داشت.</li>
    <li>Medium رفتار میانی داشت.</li>
    <li>Sparse به دلیل کم بودن نقاط core، بسیار سریع‌تر شد.</li>
  </ul>
  <p><b>نتیجه:</b> تراکم داده روی <b>زمان واقعی اجرا</b> اثر زیادی دارد.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۵. آزمایش Spatial Grid Optimization</h1>
  <p>بهترین speedup:</p>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">N = 720
ε = 0.10
MinPts = 5

Baseline = 111.411 ms
Optimized total = 5.186 ms
Speedup = 21.48×</pre>
  <p>در <code>N = 1440</code>:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline = 837.438 ms
Optimized total = 56.784 ms
Speedup = 14.75×</pre>
  <p><b>نتیجه:</b> Spatial Grid مهم‌ترین بهینه‌سازی کل پروژه بود.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۶. آزمایش Dual-Core — روش A</h1>
  <p>تشخیص <code>core-point</code> بین Core 0 و Core 1 تقسیم شد، ولی cluster expansion روی یک هسته ماند.</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline = 693.067 ms
Dual-Core = 685.656 ms
Speedup = 1.01×
Improvement = 1.07%</pre>

  <h1 style="font-size: 24px; color: #111827;">۷. آزمایش Dual-Core — روش B</h1>
  <p>در هر neighborhood query، Core 0 نصف نقاط و Core 1 نیمه دیگر را بررسی کردند.</p>
  <pre style="direction:ltr; text-align:left; background:#7f1d1d; color:#fef2f2; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline = 645.262 ms
Dual-Core = 787.090 ms
Speedup = 0.82×  ⚠️ کندتر شد</pre>
  <p><b>نتیجه:</b> هزینه synchronization بیشتر از سود parallelization شد.</p>

  <h1 style="font-size: 24px; color: #111827;">۸. آزمایش Dual-Core — روش C</h1>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Single-Core Grid = 37.085 ms
Dual-Core Grid   = 36.703 ms
Speedup = 1.01×</pre>

  <h1 style="font-size: 24px; color: #111827;">۹. آزمایش Dual-Core — روش D ⭐</h1>
  <p>در این روش خود <b>cluster expansion</b> موازی شد.</p>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Expansion speedup = 1.05×
End-to-end speedup = 1.03×
Improvement = 2.88%
Different labels = 0</pre>
  <p><b>نتیجه:</b> روش D بهترین روش Dual-Core آزمایش‌شده بود.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱۰. آزمایش Three-Sensor Simulation</h1>
  <p>یک pipeline شبیه کاربرد واقعی ساخته شد:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Sensor 1 ─┐
Sensor 2 ─┼→ Ingest/Parsing → Fusion → Spatial Grid
Sensor 3 ─┘                         ↓
                                   DBSCAN
                                     ↓
                              Human Analysis</pre>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">2 simulated humans
        ↓
2 DBSCAN clusters
        ↓
2 human-like clusters  ✅</pre>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱۱. آزمایش Real-Time Limit — 07B</h1>
  <p>بودجه هر frame در نرخ <b>10 FPS</b>:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">100 ms</pre>

  <h3 style="font-size: 18px; color: #374151;">نتیجه موفق ✅</h3>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">104 points / human / sensor
660 points / frame
6600 points / second @ 10 FPS

Average pipeline = 89.357 ms
Maximum pipeline = 96.496 ms

Detection stable = YES (2 humans)</pre>

  <h3 style="font-size: 18px; color: #374151;">اولین configuration ناموفق ❌</h3>
  <pre style="direction:ltr; text-align:left; background:#7f1d1d; color:#fef2f2; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">108 points / human / sensor
684 points / frame
Maximum pipeline = 101.247 ms
Frame overruns = 5 / 12</pre>

  <p>مرز عملی آزمایش‌شده:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">660 points/frame   ✅
684 points/frame   ❌</pre>

  <p>زمان متوسط بخش‌ها در configuration موفق:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Ingest + fusion ≈ 3.396 ms
Grid + DBSCAN   ≈ 84.708 ms
Human analysis  ≈ 1.250 ms</pre>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱۲. جمع‌بندی نهایی پروژه</h1>

  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline DBSCAN
      ↓
Parameter Analysis
      ↓
Spatial Grid
      ↓
Dual-Core Experiments
      ↓
3-Sensor Simulation
      ↓
Human Movement + Fusion
      ↓
Real-Time Limit Test</pre>

  <h3 style="font-size: 20px; color: #1d4ed8;">نتیجه اصلی پروژه</h3>
  <blockquote style="background: #eff6ff; border-right: 5px solid #3b82f6; padding: 18px 22px; border-radius: 8px; font-size: 17px; color: #1e3a8a; margin: 16px 0;">
    روی ESP32-S3، بهینه‌سازی الگوریتمی جستجوی همسایه بسیار مهم‌تر از parallelization ساده است.
    Spatial Grid بزرگ‌ترین بهبود را ایجاد می‌کند (<b>21.48×</b>) و Dual-Core زمانی مفید است که
    روی بخش محاسباتی مناسب، مانند cluster expansion، اعمال شود. در pipeline چندسنسوره شبیه‌سازی‌شده،
    سیستم فعلی در نرخ 10 FPS حداقل <b>660 نقطه در هر frame</b> را به‌صورت real-time و با تشخیص پایدار
    دو انسان پردازش می‌کند.
  </blockquote>

  <h3 style="font-size: 20px; color: #b45309;">محدودیت فعلی</h3>
  <p>
    آزمایش Real-Time هنوز با داده‌های سنسوری شبیه‌سازی‌شده انجام شده است و دریافت فیزیکی سه آنتن واقعی،
    baud rate، packet format، buffering و overhead واقعی UART هنوز وارد benchmark نشده‌اند.
    بنابراین عدد 6600 point/s یک <b>نتیجه تجربی برای pipeline شبیه‌سازی‌شده فعلی</b> است،
    نه سقف نظری نهایی ESP32-S3.
  </p>

</div>
"""

# ============ 3) نمایش در Notebook ============
display(HTML(html_content))

# ============ 4) ذخیره به فایل HTML مستقل ============
full_html = f"""<!DOCTYPE html>
<html lang="fa" dir="rtl">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>DBSCAN Benchmark Report - ESP32-S3</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Vazirmatn:wght@400;600;800&display=swap" rel="stylesheet">
<style>
  body {{
    background: #eef2f7;
    margin: 0;
    padding: 20px;
    font-family: 'Vazirmatn', 'Segoe UI', Tahoma, sans-serif;
  }}
</style>
</head>
<body>
{html_content}
</body>
</html>"""

output_path = Path("dbscan_report.html")
output_path.write_text(full_html, encoding="utf-8")

print(f"✅ گزارش با موفقیت ذخیره شد")
print(f"📁 مسیر فایل: {output_path.resolve()}")
print(f"📊 حجم فایل: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("💡 برای PDF: فایل HTML را در Chrome باز کن و Ctrl+P بزن")

✅ گزارش با موفقیت ذخیره شد
📁 مسیر فایل: C:\Users\iCFixer.ir\Documents\PlatformIO\Projects\esp32-s3-final-project-phase-1\results\dbscan_report.html
📊 حجم فایل: 14.7 KB

💡 برای PDF: فایل HTML را در Chrome باز کن و Ctrl+P بزن
